In [1]:
import os, time
from pathlib import Path
import sys
import pandas as pd
import numpy as np

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))

from src.utilities.project_paths import RAW_DIR, DATA_DIR

pd.set_option('display.float_format', '{:,.2f}'.format)

CENSUS_DIR  = RAW_DIR / 'south_africa' / 'Census2022SampleSTATA'
PARQUET_DIR = DATA_DIR / '01_interim' / 'sa_census_parquet'
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

HOUSEHOLDS_DTA = CENSUS_DIR / 'Census2022Households.dta'
PERSONS_DTA    = CENSUS_DIR / 'Census2022Persons.dta'
GEOGRAPHY_DTA  = CENSUS_DIR / 'Census2022Geography.dta'

# The Persons file is the big one (hundreds of MB). On a shared server we never
# read it in full: we take at most SAMPLE_N rows, once, via a Stata iterator.
SAMPLE_N = 100_000

# Stata value labels -> readable keys for the summary tables.
PROVINCE_LABELS = {1: 'Western Cape', 2: 'Eastern Cape', 3: 'Northern Cape',
                   4: 'Free State', 5: 'KwaZulu-Natal', 6: 'North West',
                   7: 'Gauteng', 8: 'Mpumalanga', 9: 'Limpopo'}
HHPOP_LABELS    = {1: 'Black African', 2: 'Coloured', 3: 'Indian or Asian',
                   4: 'White', 5: 'Other'}
GEOTYPE_LABELS  = {1: 'Urban area', 2: 'Tribal/traditional area', 3: 'Farm area'}

print('Census DTA dir :', CENSUS_DIR)
print('Parquet dir    :', PARQUET_DIR)
print('Source files:')
for p in (HOUSEHOLDS_DTA, PERSONS_DTA, GEOGRAPHY_DTA):
    print(f'  {p.name:28s} {p.stat().st_size / 1024**2:7.1f} MB')
print(f'Persons sample : first {SAMPLE_N:,} rows only (never the whole file)')

Census DTA dir : /Users/gabriele/App/rowsquared/py4stat-public/data/0_raw/south_africa/Census2022SampleSTATA
Parquet dir    : /Users/gabriele/App/rowsquared/py4stat-public/data/01_interim/sa_census_parquet
Source files:
  Census2022Households.dta        60.0 MB
  Census2022Persons.dta          347.0 MB
  Census2022Geography.dta         25.5 MB
Persons sample : first 100,000 rows only (never the whole file)


---
# Part A — Smart Reading: Column Selection & Dtype Hints

We use the **South Africa Census 2022 sample**, split across three Stata files
that share the household key `QID`:

| File | Grain | Size | Key columns |
|---|---|---|---|
| `Census2022Households.dta` | one row per household | ~62 MB | `DERH_HSIZE`, `DERH_HHPOP`, `H03_TENURE` |
| `Census2022Persons.dta` | one row per person | **~347 MB** | `P04_AGE`, `P02_SEX` |
| `Census2022Geography.dta` | one row per household | ~26 MB | `Province`, `District`, `Geo_type` |

**Two habits to apply:**
1. `columns=[...]` — load only what you need
2. Keep numeric codes numeric, and convert low-cardinality codes to `"category"`
   - `convert_categoricals=False` keeps Stata codes as small integers (so a size of `10`
     stays a number, not the label `"10 +"`)
   - `.astype("category")` then compresses repeating codes. For CSV/Excel, pass
     `dtype={"col": "category"}` directly to `read_csv`/`read_excel`.

## A1. Profile the full main file

In [2]:
hh_full = pd.read_stata(HOUSEHOLDS_DTA)

total_mb = hh_full.memory_usage(deep=True).sum() / 1024**2
print(f'Shape    : {hh_full.shape}')
print(f'Total RAM: {total_mb:,.1f} MB')

Shape    : (1338295, 32)
Total RAM: 121.3 MB


**Interpretation:** 1.3 million rows across 32 columns is already over 100 MB in RAM, and
the Households file is the *small* one. The Persons file has 17x more rows. The fix is always
the same — load only the columns you need.

## A2. Column-selective load with dtype conversion

Task: *"Compute mean household size by population group and tenure status."*

Columns needed: `DERH_HHPOP` (population group of head), `H03_TENURE` (tenure),
`DERH_HHSEX` (sex of head), `DERH_HSIZE` (household size), `DERH_HHAGE` (age of head).

Read with `convert_categoricals=False` so `DERH_HSIZE` stays numeric (`10` = "10 or more"),
then convert the low-cardinality demographic codes to `"category"`.

In [3]:
TASK_COLS = ['DERH_HHPOP', 'H03_TENURE', 'DERH_HHSEX', 'DERH_HSIZE', 'DERH_HHAGE']

# convert_categoricals=False keeps every column as a small integer code instead of
# pulling in the Stata string labels (which would turn DERH_HSIZE into "10 +").
hh_slim = pd.read_stata(HOUSEHOLDS_DTA, columns=TASK_COLS, convert_categoricals=False)

mb_before = hh_slim.memory_usage(deep=True).sum() / 1024**2

# Convert low-cardinality codes to category
hh_slim['DERH_HHPOP'] = hh_slim['DERH_HHPOP'].astype('category')
hh_slim['H03_TENURE'] = hh_slim['H03_TENURE'].astype('category')
hh_slim['DERH_HHSEX'] = hh_slim['DERH_HHSEX'].astype('category')

mb_after = hh_slim.memory_usage(deep=True).sum() / 1024**2
full_mb  = hh_full.memory_usage(deep=True).sum() / 1024**2

print(f'Shape      : {hh_slim.shape}')
print(f'RAM before : {mb_before:,.2f} MB')
print(f'RAM after  : {mb_after:,.2f} MB  (category encoding)')
print(f'Reduction vs full load: {full_mb / mb_after:.0f}x smaller')

Shape      : (1338295, 5)
RAM before : 7.66 MB
RAM after  : 7.66 MB  (category encoding)
Reduction vs full load: 16x smaller


**Interpretation:** Selecting 5 of 32 columns is the dominant win — roughly a 15x cut in RAM.
The codes here already arrive as 1-byte integers, so `.astype("category")` only nudges memory;
its big payoff comes with *string* codes (province names in a CSV, language labels, etc.),
where it can shrink a column 10x or more. Column selection is the cheapest optimisation — do it
by default.

## A3. Grouped analysis from the slim load

In [4]:
result = (
    hh_slim
    .groupby(['DERH_HHPOP', 'H03_TENURE'], as_index=False, observed=True)
    .agg(
        mean_hhsize  = ('DERH_HSIZE', 'mean'),
        mean_headage = ('DERH_HHAGE', 'mean'),
        n_hh         = ('DERH_HSIZE', 'count'),
    )
    .sort_values('mean_hhsize', ascending=False)
)

# map the head population-group code to its label for readability
result['DERH_HHPOP'] = result['DERH_HHPOP'].astype(int).map(HHPOP_LABELS)

display(result.head(15))

,DERH_HHPOP,H03_TENURE,mean_hhsize,mean_headage,n_hh
12,Coloured,4,3.95,53.96,31017
10,Coloured,2,3.93,50.31,4167
15,Coloured,7,3.88,47.87,815
3,Black African,4,3.82,50.42,344402
11,Coloured,3,3.79,47.27,7086
13,Coloured,5,3.69,47.28,10936
14,Coloured,6,3.64,47.70,1780
5,Black African,6,3.59,48.57,26731
4,Black African,5,3.53,47.56,237902
2,Black African,3,3.51,45.58,35360


---
# Part B — Vectorised Operations

**The rule:** reach for `.apply()` only when no vectorised form exists.

| Task | Slow | Fast |
|---|---|---|
| Conditional column | `df.apply(lambda r: ...)` | `np.where(...)` / `np.select(...)` |
| Age groups / bins | `df.apply(classify)` | `pd.cut(...)` |
| String cleaning | `df.col.apply(str.strip)` | `df.col.str.strip()` |
| Arithmetic with guard | `df.apply(lambda r: r.a/r.b if r.b else None)` | `df.a / df.b.where(df.b != 0)` |

## B1. Benchmark: `.apply()` vs `pd.cut` for age grouping

Load `P04_AGE` from the **Persons** file and classify each person into:
`"child"` (<= 14), `"working_age"` (15-64), `"elderly"` (>= 65).

We never read the whole 347 MB Persons file — we pull the first `SAMPLE_N` rows with a
Stata iterator. Measure how long `.apply()` takes vs `pd.cut`, then compute the speedup.

> **About the sample:** the file is not shuffled, so the first 100k records happen to come from a
> single province. Use this sample to demonstrate *techniques*, not to produce population estimates.

In [5]:
# Read only the first SAMPLE_N rows of the big Persons file, via an iterator.
# next(iterator) reads a single chunk and stops — the 347 MB file is never fully scanned.
with pd.read_stata(
    PERSONS_DTA,
    columns=['QID', 'PID', 'P04_AGE'],
    convert_categoricals=False,
    chunksize=SAMPLE_N,
) as it:
    persons = next(it).dropna(subset=['P04_AGE'])

print(f'Persons sampled (first {SAMPLE_N:,} rows): {len(persons):,}')

# --- .apply() ---
def age_group(age):
    if age <= 14:   return 'child'
    elif age <= 64: return 'working_age'
    else:           return 'elderly'

t0 = time.perf_counter()
persons['age_apply'] = persons['P04_AGE'].apply(age_group)
t_apply = time.perf_counter() - t0

# --- pd.cut ---
t0 = time.perf_counter()
persons['age_cut'] = pd.cut(
    persons['P04_AGE'],
    bins=[-1, 14, 64, 200],
    labels=['child', 'working_age', 'elderly'],
)
t_cut = time.perf_counter() - t0

print(f'\n.apply() : {t_apply*1000:.1f} ms')
print(f'pd.cut   : {t_cut*1000:.1f} ms')
if t_cut > 0:
    print(f'Speedup  : {t_apply/t_cut:.1f}x')

match = (persons['age_apply'] == persons['age_cut'].astype(str)).all()
print(f'\nResults match: {match}')
display(persons['age_apply'].value_counts().to_frame('apply')
        .join(persons['age_cut'].value_counts().to_frame('cut')))

Persons sampled (first 100,000 rows): 100,000

.apply() : 9.7 ms
pd.cut   : 2.4 ms
Speedup  : 4.0x

Results match: True


,apply,cut
age_apply,,
working_age,69115,69115
child,24035,24035
elderly,6850,6850


**Interpretation:** On 100k rows `pd.cut` is typically several times faster than a Python
`.apply()`, and the gap widens with data size. The pattern is what matters: use `pd.cut` for
binning, not a row-by-row Python function.

## B2. Conditional columns: `np.where` and `np.select`

Using the slim DataFrame from A2:
- `female_headed`: 1 if `DERH_HHSEX == 2` (Female), else 0  ->  use `np.where`
- `hh_size_class`: `"small"` (1-3), `"medium"` (4-6), `"large"` (7+)  ->  use `np.select`

In [6]:
# np.where -- binary condition (DERH_HHSEX: 1 = Male, 2 = Female)
hh_slim['female_headed'] = np.where(hh_slim['DERH_HHSEX'].astype(int) == 2, 1, 0)

# np.select -- multiple conditions, first match wins
conditions = [
    hh_slim['DERH_HSIZE'] <= 3,
    hh_slim['DERH_HSIZE'] <= 6,
    hh_slim['DERH_HSIZE'] > 6,
]
hh_slim['hh_size_class'] = np.select(conditions, ['small', 'medium', 'large'], default='unknown')

print('female_headed:')
display(hh_slim['female_headed'].value_counts().to_frame())

print('\nhh_size_class:')
display(hh_slim['hh_size_class'].value_counts().to_frame())

female_headed:


,count
female_headed,
0,669256
1,669039



hh_size_class:


,count
hh_size_class,
small,872802
medium,362348
large,103145


**Interpretation:** `np.where` handles the binary case cleanly. `np.select` generalises to
any number of conditions — evaluated in order, first match wins. Both operate on the whole
column at once in C, with no Python loop.

## B3. Vectorised arithmetic with a guard

Build a per-household composition from the sampled Persons rows, then compute a
**child-to-adult ratio** — `n_children / n_adults`.

Some sampled households contain only children (no adults in the sample), so the denominator
can be zero. Guard it vectorised: `n_children / n_adults.where(n_adults > 0)`.

Then show how many households have no adult (undefined ratio) and how many have a ratio > 2.

In [7]:
# Per-household counts from the 100k-row Persons sample (children <= 14, adults >= 18)
comp = (
    persons
    .assign(
        is_child=(persons['P04_AGE'] <= 14).astype(int),
        is_adult=(persons['P04_AGE'] >= 18).astype(int),
    )
    .groupby('QID', as_index=False)
    .agg(
        n_members  = ('P04_AGE', 'size'),
        n_children = ('is_child', 'sum'),
        n_adults   = ('is_adult', 'sum'),
    )
)
print(f'Households in sample: {len(comp):,}')

# Vectorised division with a zero-guard -- no .apply(), no if/else loop
comp['child_adult_ratio'] = comp['n_children'] / comp['n_adults'].where(comp['n_adults'] > 0)

no_adults = comp['child_adult_ratio'].isna().sum()
gt2       = (comp['child_adult_ratio'] > 2).sum()

print(f'No adult in sample (undefined ratio): {no_adults:,}')
print(f'Ratio > 2 (child-heavy households)  : {gt2:,}')
print()
display(comp['child_adult_ratio'].describe().to_frame())

Households in sample: 33,379
No adult in sample (undefined ratio): 41
Ratio > 2 (child-heavy households)  : 272



,child_adult_ratio
count,"33,338.00"
mean,0.37
std,0.56
min,0.00
25%,0.00
50%,0.00
75%,0.50
max,6.00


**Interpretation:** `.where(condition)` replaces values where the condition is False with NaN,
so the division produces NaN instead of dividing by zero. No `try/except`, no `.apply()`, no row
loop — the guard is applied to the whole column in one pass.

---
# Part C — Chunked Reading (a backup technique)

> **Key constraint:** accumulate `(sum, count)` per group across chunks,
> then compute `mean = sum / count` *after* the loop. "Average of averages" is wrong
> unless every chunk has exactly the same size.

## C1. Chunked mean household size by population group

Compute mean `DERH_HSIZE` per `DERH_HHPOP` using the chunk accumulator pattern.
Do **not** store rows or average per-chunk means.

Reading with `convert_categoricals=False` keeps the codes numeric and avoids the
`CategoricalConversionWarning` that Stata raises when a labelled column is read in chunks.

In [8]:
CHUNK_SIZE = 200_000   # ~7 chunks over 1.3M households
acc        = {}        # {hhpop_code: [running_sum, running_count]}
n_chunks   = 0

with pd.read_stata(
    HOUSEHOLDS_DTA,
    chunksize=CHUNK_SIZE,
    columns=['DERH_HHPOP', 'DERH_HSIZE'],
    convert_categoricals=False,
) as itr:
    for chunk in itr:
        n_chunks += 1
        clean = chunk.dropna(subset=['DERH_HHPOP', 'DERH_HSIZE'])
        for code, grp in clean.groupby('DERH_HHPOP', observed=True):
            if code not in acc:
                acc[code] = [0.0, 0]
            acc[code][0] += grp['DERH_HSIZE'].sum()
            acc[code][1] += len(grp)

chunked_means = (
    pd.DataFrame(
        [(HHPOP_LABELS.get(code, code), s / n, n) for code, (s, n) in acc.items()],
        columns=['population_group', 'mean_hhsize', 'n'],
    )
    .sort_values('mean_hhsize', ascending=False)
)

print(f'Chunks processed: {n_chunks}')
display(chunked_means)

Chunks processed: 7


,population_group,mean_hhsize,n
1,Coloured,3.25,108744
0,Black African,3.23,1050300
2,Indian or Asian,2.59,37921
3,White,2.44,137203
4,Other,2.44,4127


**Interpretation:** The accumulator pattern is correct but verbose — ~15 lines of Python for a
grouped mean. Compare to the DuckDB version in E1: a few lines of SQL, same result, no manual
loop. Use chunking when DuckDB is unavailable; otherwise prefer SQL.

## C2. What cannot be done in chunks?

Some statistics require the full dataset before computing anything. Fill in the table.

| Statistic | Chunkable? | Reason |
|---|---|---|
| Count of households per province | ✓ Yes | Counts accumulate: `total += chunk_count` |
| Global median of `DERH_HSIZE` | ✗ No | Requires all values to find the middle; (sum, count) gives the mean, not the median |
| Sum of `DERH_HSIZE` per district | ✓ Yes | Sums accumulate exactly |
| 90th percentile of `P04_AGE` | ✗ No | Percentiles require the full sorted distribution |
| Share of households with `DERH_HSIZE > 5` | ✓ Yes | Accumulate count(size > 5) and total count, then divide |

---
# Part D — File Formats: CSV, Excel, and Parquet

| Capability | CSV | Excel (.xlsx) | Parquet |
|---|---|---|---|
| Open with double-click | yes | yes | no |
| Preserves types | no | partial | yes |
| Compression | no | some | yes |
| Multiple sheets/tables | no | yes | no |
| Fast for large data | weak | no | yes |
| Read only some columns | no | weak | yes |
| Familiar to NSO staff | yes | yes | no |

## D1. Build the Parquet warehouse

Convert the Stata sources to Parquet **once**, then read them many times. We build two
analysis tables:

- `hh_main.parquet` — Households joined to Geography on `QID` (household grain, with Province)
- `persons_sample.parquet` — the **first `SAMPLE_N` rows** of the 347 MB Persons file

Because we read with `convert_categoricals=False`, every column is already a numeric code or a
plain string — there are no Stata categoricals to cast before writing Parquet.

In [9]:
HH_COLS  = ['QID', 'DERH_HSIZE', 'DERH_HHPOP', 'H03_TENURE', 'DERH_HHSEX', 'DERH_HHAGE']
GEO_COLS = ['QID', 'Province', 'District', 'Geo_type']

# 1) hh_main = Households (left) joined to Geography on QID
hh  = pd.read_stata(HOUSEHOLDS_DTA, columns=HH_COLS,  convert_categoricals=False)
geo = pd.read_stata(GEOGRAPHY_DTA,  columns=GEO_COLS, convert_categoricals=False)
hh_main = hh.merge(geo, on='QID', how='left')
hh_main.to_parquet(PARQUET_DIR / 'hh_main.parquet', index=False)

# 2) persons_sample = first SAMPLE_N rows of the big Persons file (never the whole thing)
with pd.read_stata(
    PERSONS_DTA,
    columns=['QID', 'PID', 'P02_SEX', 'P04_AGE', 'AGE_GROUP'],
    convert_categoricals=False,
    chunksize=SAMPLE_N,
) as it:
    persons_sample = next(it)
persons_sample.to_parquet(PARQUET_DIR / 'persons_sample.parquet', index=False)

# report sizes
src_mb = (HOUSEHOLDS_DTA.stat().st_size + GEOGRAPHY_DTA.stat().st_size) / 1024**2
pq_mb  = (PARQUET_DIR / 'hh_main.parquet').stat().st_size / 1024**2
print(f'hh_main.parquet        : {src_mb:6.1f} MB (2 dta files) -> {pq_mb:5.1f} MB  '
      f'({src_mb / pq_mb:.1f}x smaller)')

p_src_mb = PERSONS_DTA.stat().st_size / 1024**2
p_pq_mb  = (PARQUET_DIR / 'persons_sample.parquet').stat().st_size / 1024**2
print(f'persons_sample.parquet : first {SAMPLE_N:,} of a {p_src_mb:.0f} MB file -> '
      f'{p_pq_mb:.1f} MB  ({len(persons_sample):,} rows)')

hh_main.parquet        :   85.5 MB (2 dta files) ->  10.1 MB  (8.5x smaller)
persons_sample.parquet : first 100,000 of a 347 MB file -> 1.0 MB  (100,000 rows)


**Interpretation:** Parquet compresses each column independently — dictionary encoding for
repeated codes (province, population group), bit-packing for small integers. Census data with
many repeated codes compresses especially well. The conversion cost is paid once; every later
read is faster and smaller. Note that we **never** convert the full Persons file — only a 100k-row
sample.

## D2. Column-selective timing: Parquet vs DTA

Time reading the five household columns from Parquet vs from the original Stata file.

In [10]:
TASK_COLS = ['DERH_HSIZE', 'DERH_HHPOP', 'H03_TENURE', 'DERH_HHSEX', 'DERH_HHAGE']

t0 = time.perf_counter()
hh_pq  = pd.read_parquet(PARQUET_DIR / 'hh_main.parquet', columns=TASK_COLS)
t_pq   = time.perf_counter() - t0

t0 = time.perf_counter()
hh_dta = pd.read_stata(HOUSEHOLDS_DTA, columns=TASK_COLS, convert_categoricals=False)
t_dta  = time.perf_counter() - t0

print(f'Parquet : {t_pq:.3f}s')
print(f'Stata   : {t_dta:.3f}s')
print(f'Speedup : {t_dta / t_pq:.1f}x  (parquet faster)' if t_pq < t_dta
      else 'Note: parquet startup overhead dominated this run')

assert hh_pq.shape == hh_dta.shape
print(f'Both have shape {hh_pq.shape}')

Parquet : 0.073s
Stata   : 0.103s
Speedup : 1.4x  (parquet faster)
Both have shape (1338295, 5)


**Interpretation:** With only five narrow integer columns the two formats are close — Parquet's
pyarrow startup roughly cancels its read advantage on this shape. Parquet pulls clearly ahead when
you pick a few columns out of a *wide* file (hundreds of columns): its columnar layout reads only
those columns and skips the rest, while Stata must walk every row regardless.

## D3. Export a summary to CSV and Excel

Compute a province-level summary from `hh_main.parquet` and export it to both CSV and Excel.
These files are queried in Part E to show DuckDB working across formats.

In [11]:
CSV_PATH = PARQUET_DIR / 'province_summary.csv'
XLS_PATH = PARQUET_DIR / 'province_summary.xlsx'

province_summary = (
    pd.read_parquet(PARQUET_DIR / 'hh_main.parquet',
                    columns=['Province', 'DERH_HSIZE', 'DERH_HHAGE'])
    .dropna(subset=['Province'])
    .groupby('Province', as_index=False)
    .agg(
        mean_hhsize  = ('DERH_HSIZE', 'mean'),
        mean_headage = ('DERH_HHAGE', 'mean'),
        n_hh         = ('DERH_HSIZE', 'count'),
    )
)
province_summary.insert(1, 'province_name',
                        province_summary['Province'].astype(int).map(PROVINCE_LABELS))

province_summary.to_csv(CSV_PATH, index=False)
province_summary.to_excel(XLS_PATH, index=False)

print(f'CSV  : {CSV_PATH.stat().st_size / 1024:.1f} KB')
print(f'Excel: {XLS_PATH.stat().st_size / 1024:.1f} KB')
display(province_summary)

CSV  : 0.5 KB
Excel: 5.4 KB


,Province,province_name,mean_hhsize,mean_headage,n_hh
0,1,Western Cape,2.99,46.88,157036
1,2,Eastern Cape,3.18,49.78,151308
2,3,Northern Cape,3.42,47.36,26012
3,4,Free State,3.14,47.10,73514
4,5,KwaZulu-Natal,3.51,47.33,223374
5,6,North West,3.06,47.31,93816
6,7,Gauteng,2.78,44.16,368284
7,8,Mpumalanga,3.26,45.97,100768
8,9,Limpopo,3.43,48.14,144183


**Interpretation:** Both CSV and Excel are immediately openable by colleagues. The Excel file is
larger because of the `.xlsx` container overhead, but it preserves formatting. For a summary table
this small, either format is fine.

## D4. Conversion recipes (reference — no TODO)

Keep these for future projects:

```python
# Excel -> Parquet
pd.read_excel('src.xlsx').to_parquet('dst.parquet')

# CSV -> Parquet (with explicit types)
pd.read_csv('src.csv', dtype={'province_code': 'str'}).to_parquet('dst.parquet')

# Parquet -> Excel (for stakeholders)
pd.read_parquet('data.parquet').to_excel('report.xlsx', index=False)

# Parquet -> CSV (for sharing)
pd.read_parquet('data.parquet').to_csv('export.csv', index=False)
```

---
# Part E — DuckDB: SQL Across Formats

| Situation | Tool |
|---|---|
| Aggregations over large files | DuckDB |
| Joining files of different formats | DuckDB |
| Reading only a few columns from a wide Parquet file | DuckDB |
| Statistical modelling | Pandas (after DuckDB preprocessing) |
| Quick in-memory DataFrame manipulation | Pandas |

In [12]:
try:
    import duckdb
    print(f'duckdb {duckdb.__version__} ready')
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'duckdb', '-q'])
    import duckdb
    print(f'duckdb {duckdb.__version__} installed')

HH_PARQUET      = str(PARQUET_DIR / 'hh_main.parquet')
PERSONS_PARQUET = str(PARQUET_DIR / 'persons_sample.parquet')
CSV_PATH_STR    = str(CSV_PATH)
XLS_PATH_STR    = str(XLS_PATH)

duckdb 1.5.4 ready


## E1. Query Parquet — grouped aggregation

Replicate a grouped mean using a single SQL query on Parquet — no Python loop, no chunking.

In [13]:
result = duckdb.sql(f"""
    SELECT
        Province,
        AVG(DERH_HSIZE) AS mean_hhsize,
        COUNT(*)        AS n_hh
    FROM read_parquet('{HH_PARQUET}')
    WHERE Province   IS NOT NULL
      AND DERH_HSIZE IS NOT NULL
    GROUP BY Province
    ORDER BY mean_hhsize DESC
""").to_df()

result['province'] = result['Province'].astype(int).map(PROVINCE_LABELS)
display(result)

,Province,mean_hhsize,n_hh,province
0,5,3.51,223374,KwaZulu-Natal
1,9,3.43,144183,Limpopo
2,3,3.42,26012,Northern Cape
3,8,3.26,100768,Mpumalanga
4,2,3.18,151308,Eastern Cape
5,4,3.14,73514,Free State
6,6,3.06,93816,North West
7,1,2.99,157036,Western Cape
8,7,2.78,368284,Gauteng


**Interpretation:** DuckDB internally chunks the Parquet file and aggregates — but you wrote a
few lines of SQL instead of the 15-line Python accumulator from C1. For grouped aggregations, SQL
is almost always the right tool.

## E2. JOIN across Parquet files

Join `hh_main.parquet` (one row per household) to `persons_sample.parquet` (one row per person)
on `QID`, then compute **mean age (`P04_AGE`) by the household head's population group**
(`DERH_HHPOP`), with household and member counts.

> **Why group by population group, not province?** `persons_sample.parquet` is the first 100k
> records of the file, which all fall in one province — so province would give a single row.
> Population group varies within the sample, so it shows the join working. Treat this as a
> mechanics demo, not a population estimate.

> **Sanity check:** more members than households is expected (many members per household).

In [14]:
group_age = duckdb.sql(f"""
    SELECT
        hh.DERH_HHPOP,
        AVG(p.P04_AGE)          AS mean_age,
        COUNT(DISTINCT hh.QID)  AS n_households,
        COUNT(*)                AS n_members
    FROM   read_parquet('{HH_PARQUET}')      AS hh
    JOIN   read_parquet('{PERSONS_PARQUET}') AS p
        ON hh.QID = p.QID
    WHERE  p.P04_AGE    IS NOT NULL
      AND  hh.DERH_HHPOP IS NOT NULL
    GROUP  BY hh.DERH_HHPOP
    ORDER  BY mean_age DESC
""").to_df()

group_age['population_group'] = group_age['DERH_HHPOP'].astype(int).map(HHPOP_LABELS)
display(group_age)
print(f'\nSanity: n_members >= n_households always? '
      f'{(group_age["n_members"] >= group_age["n_households"]).all()}')

,DERH_HHPOP,mean_age,n_households,n_members,population_group
0,4,40.88,7749,19034,White
1,5,36.05,332,898,Other
2,3,34.14,810,2249,Indian or Asian
3,2,32.33,10307,38029,Coloured
4,1,29.13,14181,39790,Black African



Sanity: n_members >= n_households always? True


**Interpretation:** DuckDB executes the join without loading either Parquet file fully into
Python memory — it reads only the columns it needs and streams the join. A `pd.merge()` would
require both files in RAM at once.

## E3. Query CSV directly

DuckDB can query a CSV on disk without loading it into Python first. Query the province summary
CSV from D3 — no `pd.read_csv()` needed.

In [15]:
result_csv = duckdb.sql(f"""
    SELECT *
    FROM   '{CSV_PATH_STR}'
    ORDER  BY mean_hhsize DESC
""").to_df()

display(result_csv)
print('DuckDB read the CSV directly; no pd.read_csv() was needed.')

,Province,province_name,mean_hhsize,mean_headage,n_hh
0,5,KwaZulu-Natal,3.51,47.33,223374
1,9,Limpopo,3.43,48.14,144183
2,3,Northern Cape,3.42,47.36,26012
3,8,Mpumalanga,3.26,45.97,100768
4,2,Eastern Cape,3.18,49.78,151308
5,4,Free State,3.14,47.10,73514
6,6,North West,3.06,47.31,93816
7,1,Western Cape,2.99,46.88,157036
8,7,Gauteng,2.78,44.16,368284


DuckDB read the CSV directly; no pd.read_csv() was needed.


**Interpretation:** DuckDB auto-detects the delimiter and column types from the CSV header. For
larger CSVs, `read_csv_auto('path', all_varchar=true)` reads everything as strings first so codes
like `'001'` are not silently coerced.

## E4. Join formats: Parquet + Excel

DuckDB can join different formats in one query. Use the Excel file from D3 as a lookup table and
join it to the Parquet survey file.

**Approach:** read Excel with pandas -> register as a DuckDB view -> join in SQL.

In [16]:
province_lookup = pd.read_excel(XLS_PATH)
duckdb.register('province_lookup', province_lookup)

result_join = duckdb.sql(f"""
    SELECT
        hh.Province,
        pl.mean_hhsize       AS benchmark_hhsize,
        AVG(hh.DERH_HSIZE)   AS actual_hhsize,
        COUNT(*)             AS n_hh
    FROM   read_parquet('{HH_PARQUET}') AS hh
    JOIN   province_lookup              AS pl
        ON hh.Province = pl.Province
    GROUP  BY hh.Province, pl.mean_hhsize
    ORDER  BY hh.Province
""").to_df()

display(result_join)
diff = (result_join['benchmark_hhsize'] - result_join['actual_hhsize']).abs().max()
print(f'\nMax diff (benchmark vs actual): {diff:.2e}  <- same underlying data, should be ~0')

,Province,benchmark_hhsize,actual_hhsize,n_hh
0,1,2.99,2.99,157036
1,2,3.18,3.18,151308
2,3,3.42,3.42,26012
3,4,3.14,3.14,73514
4,5,3.51,3.51,223374
5,6,3.06,3.06,93816
6,7,2.78,2.78,368284
7,8,3.26,3.26,100768
8,9,3.43,3.43,144183



Max diff (benchmark vs actual): 4.44e-16  <- same underlying data, should be ~0


**Interpretation:** One engine, two formats joined together. The pattern extends to any
combination — Stata (via pandas), CSV, Parquet, and even an API response registered as a
DataFrame view. SQL is the contract; DuckDB handles the format differences.

## E5. Persist a result as Parquet with `COPY TO`

DuckDB can write query results straight to Parquet, bypassing Python memory.

```sql
COPY (...query...) TO 'output.parquet' (FORMAT PARQUET);
```

Write the population-group age summary from E2 to
`PARQUET_DIR / 'group_age_summary.parquet'`, then read it back with pandas to verify.

In [17]:
OUT_PATH = str(PARQUET_DIR / 'group_age_summary.parquet')

duckdb.sql(f"""
    COPY (
        SELECT
            hh.DERH_HHPOP,
            AVG(p.P04_AGE)          AS mean_age,
            COUNT(DISTINCT hh.QID)  AS n_households,
            COUNT(*)                AS n_members
        FROM   read_parquet('{HH_PARQUET}')      AS hh
        JOIN   read_parquet('{PERSONS_PARQUET}') AS p
            ON hh.QID = p.QID
        WHERE  p.P04_AGE    IS NOT NULL
          AND  hh.DERH_HHPOP IS NOT NULL
        GROUP  BY hh.DERH_HHPOP
        ORDER  BY mean_age DESC
    ) TO '{OUT_PATH}' (FORMAT PARQUET)
""")

summary = pd.read_parquet(OUT_PATH)
print(f'Written to: {OUT_PATH}')
print(f'Shape     : {summary.shape}')
display(summary)

Written to: /Users/gabriele/App/rowsquared/py4stat-public/data/01_interim/sa_census_parquet/group_age_summary.parquet
Shape     : (5, 4)


,DERH_HHPOP,mean_age,n_households,n_members
0,4,40.88,7749,19034
1,5,36.05,332,898
2,3,34.14,810,2249
3,2,32.33,10307,38029
4,1,29.13,14181,39790


**Interpretation:** `COPY ... TO` bypasses Python for the write step — DuckDB streams the query
result straight to disk. On a multi-gigabyte intermediate result this avoids allocating a large
DataFrame just to call `.to_parquet()` on it. The output is readable by any Parquet tool: pandas,
R, Spark, DuckDB in another session.

---
# Summary — When to use each technique

| Situation | Recommended approach |
|---|---|
| Wide file, need 5 of 32 columns | `pd.read_stata(columns=[...])` or `pd.read_parquet(columns=[...])` |
| A 347 MB file on a shared server | Read a capped sample once via a `chunksize` iterator |
| Keeping survey codes numeric | `convert_categoricals=False`, then `.astype('category')` |
| Classify ages into groups | `pd.cut(df['P04_AGE'], bins=[...], labels=[...])` |
| Same file queried many times | Convert to Parquet once, read Parquet every time |
| File larger than RAM, need a grouped mean | Chunk accumulator (sum + count per group) — or DuckDB |
| JOIN two large files without loading either | DuckDB `read_parquet()` with SQL JOIN |
| Query a CSV without loading it | `duckdb.sql(f"SELECT ... FROM '{path}'")` |
| Share results with a non-Python colleague | CSV (universal) or Excel (familiar to NSO staff) |